# Integrated Advanced Training, Tokenization, and Knowledge Distillation Notebook

This notebook demonstrates how to merge multiple advanced training techniques into one workflow. We cover:

- **Teacher–Student Knowledge Distillation** using a simple classification model
- **Parallelization** using Python’s multiprocessing/threading for data preparation and training routines
- **Memory-Efficient Techniques** including gradient checkpointing, mixed precision training, and efficient attention
- **Advanced GPT Model Definitions** with custom layers, ALiBi bias, and a multihead latent attention module
- **Teacher Data Generation and Tokenization** using Hugging Face’s tokenizer and datasets
- **Full Training Pipeline** with checkpointing, logging, and hardware-specific optimizations

At the end, two training routines (the simple distillation task and the advanced GPT training) are launched in parallel.

In [ ]:
# Device Setup and Reproducibility
import os
import time
import math
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm

# Setting up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Setup logging
logging.basicConfig(
    filename='training_log.txt',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger()

In [ ]:
# ----------------------------------------------
# Section 1: Simple Teacher–Student Distillation
# ----------------------------------------------

import torch.optim as optim

class TeacherModel(nn.Module):
    def __init__(self, input_dim=20, hidden_dim=50, output_dim=5):
        super(TeacherModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return self.net(x)

class StudentModel(nn.Module):
    def __init__(self, input_dim=20, hidden_dim=20, output_dim=5):
        super(StudentModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return self.net(x)

def run_teacher_distillation():
    """
    A simple distillation loop where a student model learns from a teacher’s softened outputs.
    """
    torch.manual_seed(0)
    teacher = TeacherModel()
    student = StudentModel()

    # Generate dummy data
    X = torch.randn(100, 20)  # 100 samples, 20 features each

    # Get teacher predictions (soft targets)
    teacher.eval()
    with torch.no_grad():
        teacher_logits = teacher(X)
    soft_targets = torch.softmax(teacher_logits, dim=1)

    # Knowledge distillation training loop for student
    student.train()
    optimizer = optim.SGD(student.parameters(), lr=0.1)
    loss_fn = nn.KLDivLoss(reduction='batchmean')
    for epoch in range(5):
        student_logits = student(X)
        student_log_probs = torch.log_softmax(student_logits, dim=1)
        loss = loss_fn(student_log_probs, soft_targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 2 == 0:
            print(f"Teacher Distillation Epoch {epoch}: loss = {loss.item():.4f}")

# You can run this simple distillation independently or in parallel with the advanced training below.

In [ ]:
# ----------------------------------------------
# Section 2: Parallel Processing Demo & Advanced Techniques
# ----------------------------------------------

import multiprocessing

# Dummy parallel processing using a CPU-bound function
data = list(range(1, 21))

def slow_double(x):
    total = 0
    for i in range(100000):
        total += x * 2
    return total

pool = multiprocessing.Pool(processes=4)
results = pool.map(slow_double, data)
pool.close()
pool.join()
print("Parallel computation results (first 5):", results[:5])

In [ ]:
# Gradient Checkpointing and Accumulation Demo
from torch.utils.checkpoint import checkpoint

class SimpleModel(nn.Module):
    def __init__(self, layer_sizes=[128, 128, 128, 128]):
        super(SimpleModel, self).__init__()
        self.layers = nn.ModuleList([nn.Linear(in_f, out_f) for in_f, out_f in zip(layer_sizes, layer_sizes[1:])])
    def forward(self, x):
        def run_layers(start, end, inp):
            out = inp
            for layer in self.layers[start:end]:
                out = torch.relu(layer(out))
            return out
        x = torch.relu(self.layers[0](x))
        x = checkpoint(lambda inp: run_layers(1, 3, inp), x)
        x = torch.relu(self.layers[3](x))
        return x

model_checkpoint = SimpleModel([16, 16, 16, 16])
optimizer_cp = optim.SGD(model_checkpoint.parameters(), lr=0.01)
inp_cp = torch.randn(4, 16, requires_grad=True)
out_cp = model_checkpoint(inp_cp)
loss_cp = out_cp.sum()
loss_cp.backward()
print("Gradient checkpointing applied, backward computed successfully.")

# Gradient accumulation example
model_acc = nn.Linear(10, 1)
optimizer_acc = optim.SGD(model_acc.parameters(), lr=0.1)
batch_size = 32
accumulate_steps = 4
for step in range(accumulate_steps):
    batch_x = torch.randn(batch_size//accumulate_steps, 10)
    batch_y = torch.randn(batch_size//accumulate_steps, 1)
    out_acc = model_acc(batch_x)
    loss_acc = nn.MSELoss()(out_acc, batch_y)
    loss_acc.backward()
    if (step + 1) % accumulate_steps == 0:
        optimizer_acc.step()
        optimizer_acc.zero_grad()
        print(f"Performed optimizer step at mini-batch {step+1}")

In [ ]:
# Dynamic Programming for Efficient Attention Demo using PyTorch's optimized API
batch, heads, seq_len, dim = 1, 1, 10, 16
query = torch.randn(batch, heads, seq_len, dim)
key   = torch.randn(batch, heads, seq_len, dim)
value = torch.randn(batch, heads, seq_len, dim)
attn_output = F.scaled_dot_product_attention(query, key, value, dropout_p=0.0, is_causal=False)
print("Attention output shape:", attn_output.shape)

In [ ]:
# Mixed Precision Training Demo
import torch.cuda.amp as amp

# Reuse the StudentModel from above for demonstration
model_mp = StudentModel()
data_mp = torch.randn(10, 20)
with torch.no_grad():
    teacher_logits_mp = TeacherModel()(data_mp)
soft_targets_mp = torch.softmax(teacher_logits_mp, dim=1)
optimizer_mp = optim.SGD(model_mp.parameters(), lr=0.01)
scaler = amp.GradScaler(enabled=(device.type != 'cpu'))

# Move to MPS if available
if torch.backends.mps.is_available():
    device_mp = torch.device('mps')
    model_mp.to(device_mp)
    data_mp = data_mp.to(device_mp)
    soft_targets_mp = soft_targets_mp.to(device_mp)
else:
    device_mp = device

model_mp.train()
optimizer_mp.zero_grad()
with amp.autocast(device_type=device_mp.type, dtype=torch.float16, enabled=(device_mp.type != 'cpu')):
    outputs_mp = model_mp(data_mp)
    loss_mp = nn.KLDivLoss(reduction='batchmean')(torch.log_softmax(outputs_mp, dim=1), soft_targets_mp)
scaler.scale(loss_mp).backward()
scaler.step(optimizer_mp)
scaler.update()
print(f"Loss (mixed precision training step): {loss_mp.item():.4f}")

In [ ]:
# Efficient Dataset Tokenization and Batching Demo
from torch.nn.utils.rnn import pad_sequence

# Dummy tokenized sequences (varying lengths)
tokenized_sequences = [
    torch.tensor([1, 5, 8, 2]),         
    torch.tensor([3, 3, 3]),            
    torch.tensor([7, 9, 10, 11, 2, 2])  
]
tokenized_sequences.sort(key=lambda x: -x.size(0))
padded_batch = pad_sequence(tokenized_sequences, batch_first=True, padding_value=0)
print("Padded batch shape:", padded_batch.shape)
print("Padded batch contents:\n", padded_batch)

## Advanced GPT Model Definitions and Training Setup

In [ ]:
# Advanced Model Definitions (GPTConfig, Transformer, and Enhanced GPTLMHeadModel)
import sympy as sp

def chunked_matmul(a, b, chunk_size=64):
    B, n_heads, T, head_dim = a.shape
    output_chunks = []
    for start in range(0, T, chunk_size):
        end = min(start + chunk_size, T)
        chunk_result = torch.matmul(a[:, :, start:end, :], b)  
        output_chunks.append(chunk_result)
    return torch.cat(output_chunks, dim=2)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-4):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm_x = x.norm(2, dim=-1, keepdim=True)
        rms_x = norm_x / math.sqrt(x.shape[-1])
        return (x / (rms_x + self.eps)) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, hidden_dim, expansion_factor=4, dropout_prob=0.1):
        super().__init__()
        self.expanded_dim = expansion_factor * hidden_dim
        self.fc_in = nn.Linear(hidden_dim, 2 * self.expanded_dim)
        self.fc_out = nn.Linear(self.expanded_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout_prob)
    def forward(self, x):
        x_proj = self.fc_in(x)
        x1, x2 = x_proj.chunk(2, dim=-1)
        x_out = F.silu(x1) * x2
        x_out = self.fc_out(x_out)
        return self.dropout(x_out)

def build_alibi_tensor(batch_size, n_heads, seq_len, device):
    def get_slopes(n):
        def get_slopes_power_of_2(n):
            start = 2 ** (-8.0 / n)
            ratio = start
            return [start * (ratio ** i) for i in range(n)]
        if math.log2(n).is_integer():
            return get_slopes_power_of_2(n)
        else:
            closest_power_of_2 = 2 ** math.ceil(math.log2(n))
            slopes = get_slopes_power_of_2(closest_power_of_2)
            return slopes[:n]
    slopes = torch.tensor(get_slopes(n_heads), device=device).unsqueeze(-1).unsqueeze(-1)
    arange_tensor = torch.arange(seq_len, device=device).unsqueeze(0).unsqueeze(0)
    alibi = slopes * arange_tensor
    return alibi

class GPTConfig:
    def __init__(self, vocab_size, max_seq_len, n_embd, n_layer, n_head, dropout_prob=0.1, alibi=True, flash_attention=True):
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.n_embd = n_embd
        self.n_layer = n_layer
        self.n_head = n_head
        self.dropout_prob = dropout_prob
        self.alibi = alibi
        self.flash_attention = flash_attention

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        assert self.n_embd % self.n_head == 0, "Embedding dim must be divisible by number of heads"
        self.head_dim = self.n_embd // self.n_head
        self.q_proj = nn.Linear(self.n_embd, self.n_embd)
        self.k_proj = nn.Linear(self.n_embd, self.n_embd)
        self.v_proj = nn.Linear(self.n_embd, self.n_embd)
        self.out_proj = nn.Linear(self.n_embd, self.n_embd)
        self.dropout = nn.Dropout(config.dropout_prob)
        self.flash_attention = config.flash_attention
        self.register_buffer("causal_mask", torch.tril(torch.ones(config.max_seq_len, config.max_seq_len)), persistent=False)
    def forward(self, x, attention_mask=None, alibi_bias=None):
        B, T, C = x.size()
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if self.flash_attention and alibi_bias is None and attention_mask is None:
            attn_output = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout.p, is_causal=True)
        else:
            attn_scores = chunked_matmul(q, k.transpose(-2, -1), chunk_size=64) / math.sqrt(self.head_dim)
            causal_mask = self.causal_mask[:T, :T]
            attn_scores = attn_scores.masked_fill(causal_mask == 0, float('-inf'))
            if alibi_bias is not None:
                attn_scores = attn_scores + alibi_bias[:, :, :T]
            if attention_mask is not None:
                extended_mask = attention_mask.unsqueeze(1).unsqueeze(2)
                attn_scores = attn_scores.masked_fill(extended_mask == 0, float('-inf'))
            attn_weights = F.softmax(attn_scores, dim=-1)
            attn_weights = self.dropout(attn_weights)
            attn_output = torch.matmul(attn_weights, v)
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, C)
        output = self.out_proj(attn_output)
        return self.dropout(output)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attn_norm = RMSNorm(config.n_embd)
        self.ffn_norm = RMSNorm(config.n_embd)
        self.attn = MultiHeadSelfAttention(config)
        self.mlp = SwiGLU(config.n_embd, expansion_factor=4, dropout_prob=config.dropout_prob)
        self.dropout = nn.Dropout(config.dropout_prob)
    def forward(self, x, attention_mask=None, alibi_bias=None):
        normed_x = self.attn_norm(x)
        attn_out = self.attn(normed_x, attention_mask=attention_mask, alibi_bias=alibi_bias)
        x = x + attn_out
        normed_x2 = self.ffn_norm(x)
        ffn_out = self.mlp(normed_x2)
        return x + ffn_out

class GPTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.vocab_size = config.vocab_size
        self.max_seq_len = config.max_seq_len
        self.n_embd = config.n_embd
        self.n_layer = config.n_layer
        self.n_head = config.n_head
        self.wte = nn.Embedding(self.vocab_size, self.n_embd)
        self.drop = nn.Dropout(config.dropout_prob)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(self.n_layer)])
        self.norm_f = RMSNorm(config.n_embd)
        self.apply(self._init_weights)
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0, std=0.02)
    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        assert T <= self.max_seq_len, "Sequence length exceeds model's maximum."
        token_embeddings = self.wte(input_ids)
        x = self.drop(token_embeddings)
        alibi_bias = None
        if self.config.alibi:
            alibi_bias = build_alibi_tensor(B, self.n_head, T, device=x.device)
        for block in self.blocks:
            x = block(x, attention_mask=attention_mask, alibi_bias=alibi_bias)
        return self.norm_f(x)

class GPTLMHeadModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.transformer = GPTModel(config)
        self.vocab_size = config.vocab_size
        self.n_embd = config.n_embd
    def forward(self, input_ids, attention_mask=None, labels=None):
        hidden_states = self.transformer(input_ids, attention_mask=attention_mask)
        logits = F.linear(hidden_states, self.transformer.wte.weight)
        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = nn.CrossEntropyLoss()(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        return {"loss": loss, "logits": logits}

class MLA(nn.Module):
    def __init__(self, n_embd, n_latent, n_head, dropout_prob=0.1):
        super().__init__()
        self.n_latent = n_latent
        self.latent = nn.Parameter(torch.randn(n_latent, n_embd))
        self.q_proj = nn.Linear(n_embd, n_embd)
        self.k_proj = nn.Linear(n_embd, n_embd)
        self.v_proj = nn.Linear(n_embd, n_embd)
        self.out_proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout_prob)
        self.n_head = n_head
        self.head_dim = n_embd // n_head
    def forward(self, x):
        B, T, C = x.size()
        latent = self.latent.unsqueeze(0).expand(B, self.n_latent, C)
        q = self.q_proj(latent)
        k = self.k_proj(x)
        v = self.v_proj(x)
        q = q.view(B, self.n_latent, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        latent_out = torch.matmul(attn_weights, v)
        latent_out = latent_out.transpose(1, 2).contiguous().view(B, self.n_latent, C)
        aggregated = latent_out.mean(dim=1)
        enhanced = x + self.out_proj(aggregated).unsqueeze(1).expand(-1, T, -1)
        return enhanced

class EnhancedGPTLMHeadModel(nn.Module):
    def __init__(self, config, mla_n_latent=16, use_mla=True, use_calculator=False):
        super().__init__()
        self.gpt_lm_head = GPTLMHeadModel(config)
        self.use_mla = use_mla
        if self.use_mla:
            self.mla = MLA(config.n_embd, n_latent=mla_n_latent, n_head=config.n_head, dropout_prob=config.dropout_prob)
        self.use_calculator = use_calculator
        if self.use_calculator:
            self.calculator = CalculatorTool()
    def forward(self, input_ids, attention_mask=None, labels=None):
        output = self.gpt_lm_head(input_ids, attention_mask=attention_mask, labels=labels)
        hidden_states = self.gpt_lm_head.transformer(input_ids, attention_mask=attention_mask)
        hidden_states = self.gpt_lm_head.transformer.norm_f(hidden_states)
        if self.use_mla:
            hidden_states = self.mla(hidden_states)
            logits = F.linear(hidden_states, self.gpt_lm_head.transformer.wte.weight)
            output["logits"] = logits
        return output
    def calculate(self, expression: str) -> str:
        if self.use_calculator:
            return self.calculator.calculate(expression)
        else:
            return "Calculator tool not enabled." 

class CalculatorTool:
    def __init__(self):
        pass
    def calculate(self, expression: str) -> str:
        try:
            expr = sp.sympify(expression)
            result = sp.nsimplify(expr)
            return str(result)
        except Exception as e:
            return f"Error: {e}"

In [ ]:
# Teacher Data Generation using external command (e.g., 'ollama run tinyllama')
import subprocess
import random

def generate_teacher_sample(prompt):
    command = ["ollama", "run", "tinyllama"]
    try:
        output = subprocess.check_output(command, input=prompt, text=True)
        return output.strip()
    except subprocess.CalledProcessError as e:
        print(f"Error generating sample for prompt: {prompt}. Error: {e}")
        return None

seed_prompts = [
    "Once upon a time in a futuristic city,",
    "In the distant past, a forgotten civilization",
    "A scientist discovers a formula that",
    "The AI consciousness awakens and realizes",
    "The secrets of the universe are hidden in",
    "A lone adventurer stumbles upon a hidden cave",
    "A mathematician solves an unsolvable problem",
    "In the heart of a digital simulation, an entity",
    "The last human on Earth writes a message to",
    "The world changes forever when a new species emerges",
    "A rogue time traveler disrupts history by",
    "On a distant planet, a civilization communicates using",
    "An ancient book reveals the truth about",
    "The future of robotics takes a turn when",
    "A cybernetic mind explores the depths of",
    "A secret society guards the knowledge of",
    "The experiment went terribly wrong when",
    "A quantum physicist uncovers a paradox that",
    "The battle between humans and AI begins when",
    "A virtual world becomes indistinguishable from reality",
    "A lost AI finds itself wandering across time",
    "A hacker deciphers a message from an unknown source",
    "The world's most powerful AI makes a decision that",
    "An anomaly in space leads to a new discovery",
    "A mysterious signal from deep space changes everything"
]

samples = []
max_samples = 100

for _ in range(max_samples):
    prompt = random.choice(seed_prompts)
    teacher_output = generate_teacher_sample(prompt)
    if teacher_output:
        samples.append({"text": teacher_output})
        print(f"Generated sample {_+1}: {teacher_output[:100]}...")
        time.sleep(0.5)
    if teacher_output and len(samples) % 5 == 0:
        new_prompt = teacher_output[:50]
        seed_prompts.append(new_prompt)

print(f"Generated {len(samples)} training samples from teacher model.")

In [ ]:
# Data Tokenization using GPT2 Tokenizer and Hugging Face Datasets
from transformers import GPT2Tokenizer
from datasets import Dataset

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

teacher_dataset = Dataset.from_list(samples)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

tokenized_dataset = teacher_dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])

train_dataloader = DataLoader(tokenized_dataset, batch_size=8, shuffle=True)

print('Teacher-generated data successfully loaded, tokenized, and prepared for training!')

In [ ]:
# Model Initialization
config = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    max_seq_len=1024,
    n_embd=512,
    n_layer=10,
    n_head=8,
    dropout_prob=0.05,
    alibi=True,
    flash_attention=True
)

model = EnhancedGPTLMHeadModel(
    config, 
    mla_n_latent=16,
    use_mla=True,
    use_calculator=True
).to(device)

if device.type != 'cuda' and hasattr(torch, 'compile'):
    print("Compiling model for CPU acceleration using torch.compile (DeepSeek optimization activated)!")
    try:
        model = torch.compile(model, mode='reduce-overhead')
    except Exception as e:
        print("Compilation failed, continuing without compile. Error:", e)

total_params = sum(p.numel() for p in model.parameters())
print(f'Total Parameters: {total_params}')
print('Enhanced Model initialized successfully.')

In [ ]:
from torch.optim import AdamW

# Optimizer and Scheduler Setup
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

num_epochs = 3
num_training_steps = len(train_dataloader) * num_epochs
num_warmup_steps = 0

def lr_lambda(current_step):
    if current_step < num_warmup_steps:
        return float(current_step) / float(max(1, num_warmup_steps))
    return max(0.0, float(num_training_steps - current_step) / float(max(1, num_training_steps - num_warmup_steps)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

In [ ]:
def load_latest_checkpoint(model, optimizer, scheduler, checkpoint_dir='./checkpoints_turbo'):
    if not os.path.exists(checkpoint_dir):
        print(f"No checkpoint directory found at '{checkpoint_dir}'. Starting training from scratch.")
        return model, optimizer, scheduler, 0, 0
    checkpoints = [os.path.join(checkpoint_dir, ckpt) for ckpt in os.listdir(checkpoint_dir) if ckpt.endswith('.pt')]
    if not checkpoints:
        print(f"No checkpoints found in '{checkpoint_dir}'. Starting training from scratch.")
        return model, optimizer, scheduler, 0, 0
    latest_ckpt = max(checkpoints, key=os.path.getctime)
    checkpoint = torch.load(latest_ckpt, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'], strict=False)
    try:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    except ValueError as e:
        print(f"Warning: Optimizer state dict mismatch - {e}. Skipping optimizer state load.")
    try:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    except ValueError as e:
        print(f"Warning: Scheduler state dict mismatch - {e}. Skipping scheduler state load.")
    epoch = checkpoint['epoch']
    global_step = checkpoint['global_step']
    print(f"Loaded checkpoint '{latest_ckpt}' from epoch {epoch+1}, step {global_step}.")
    return model, optimizer, scheduler, epoch + 1, global_step

model, optimizer, scheduler, start_epoch, global_step = load_latest_checkpoint(model, optimizer, scheduler)

In [ ]:
# ----------------------------------------------
# Section 3: Parallel Training Routines
# ----------------------------------------------

def run_gpt_training():
    """
    Advanced GPT training loop with checkpointing, validation, and logging.
    """
    epochs = num_epochs
    checkpoint_interval = 600  # seconds
    global_step_local = global_step
    last_checkpoint_time = time.time()
    checkpoint_dir = './checkpoints_turbo'

    print('Starting GPT training...')
    model.train()
    for epoch in range(start_epoch, epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        epoch_iterator = tqdm(train_dataloader, desc="GPT Training")
        for batch in epoch_iterator:
            inputs = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=inputs, attention_mask=attention_mask, labels=inputs)
            loss = outputs["loss"]
            if torch.isnan(loss):
                print(f"NaN loss encountered at step {global_step_local}. Skipping update.")
                logger.warning(f"NaN loss at step {global_step_local}")
                optimizer.zero_grad()
                continue
            try:
                with torch.autograd.detect_anomaly():
                    loss.backward()
            except RuntimeError as e:
                print(f"Runtime error during backward pass at step {global_step_local}: {e}")
                logger.error(f"Backward error at step {global_step_local}: {e}")
                optimizer.zero_grad()
                continue
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step_local += 1
            epoch_iterator.set_postfix(loss=loss.item())
            if time.time() - last_checkpoint_time >= checkpoint_interval:
                os.makedirs(checkpoint_dir, exist_ok=True)
                ckpt_path = os.path.join(checkpoint_dir, f'checkpoint-epoch{epoch+1}-step{global_step_local}.pt')
                torch.save({
                    'epoch': epoch,
                    'global_step': global_step_local,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict()
                }, ckpt_path)
                print(f"Saved checkpoint at step {global_step_local}")
                logger.info(f"Saved checkpoint at step {global_step_local}")
                last_checkpoint_time = time.time()
        model.eval()
        val_losses = []
        generated_outputs = []
        expected_outputs = []
        with torch.no_grad():
            for batch in train_dataloader:
                inputs = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                outputs = model(input_ids=inputs, attention_mask=attention_mask, labels=inputs)
                val_losses.append(outputs["loss"].item())
                sample_input = inputs[0:1]
                generated_ids = sample_input
                for _ in range(50):
                    logits = F.linear(model.transformer(sample_input), model.transformer.wte.weight)
                    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                    generated_ids = torch.cat((generated_ids, next_token), dim=1)
                generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
                expected_text = tokenizer.decode(inputs[0], skip_special_tokens=True)
                generated_outputs.append(generated_text)
                expected_outputs.append(expected_text)
        avg_val_loss = sum(val_losses) / len(val_losses)
        print(f"Validation Loss after epoch {epoch+1}: {avg_val_loss}")
        logger.info(f"Epoch {epoch+1} - Validation Loss: {avg_val_loss}")
        for i, (gen, exp) in enumerate(zip(generated_outputs[:3], expected_outputs[:3])):
            log_str = f"Sample {i+1}:\nExpected: {exp}\nGenerated: {gen}\n{'-'*20}"
            print(log_str)
            logger.info(log_str)
        model.train()
    print('GPT Training complete!')

# End of run_gpt_training

In [ ]:
# ----------------------------------------------
# Section 4: Launch Parallel Training
# ----------------------------------------------

import threading

# Create threads for the two training routines
teacher_thread = threading.Thread(target=run_teacher_distillation)
gpt_thread = threading.Thread(target=run_gpt_training)

print('Starting parallel training: Teacher distillation and GPT training.')
teacher_thread.start()
gpt_thread.start()

teacher_thread.join()
gpt_thread.join()

print('Both training routines have completed.')

In [ ]:
# Checkpoint Generation and Text Generation Demo
def generate_text(prompt, model, tokenizer, max_length=50, temperature=0.7, top_k=50):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    generated = input_ids
    with torch.no_grad():
        for _ in range(max_length - input_ids.size(1)):
            outputs = model(generated)
            logits = outputs['logits'][:, -1, :] / temperature
            values, _ = torch.topk(logits, k=top_k, dim=-1)
            min_values = values[:, -1].unsqueeze(1)
            filtered_logits = torch.where(logits < min_values, torch.full_like(logits, float('-inf')), logits)
            probabilities = F.softmax(filtered_logits, dim=-1)
            next_token = torch.multinomial(probabilities, num_samples=1)
            generated = torch.cat((generated, next_token), dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break
    return tokenizer.decode(generated[0], skip_special_tokens=True)

prompt = "Once upon a time, there was a girl named Alice who discovered a mysterious key"
print("Prompt:", prompt)
print("Generated Text:", generate_text(prompt, model, tokenizer))